## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")


In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)

/home/common/gen-ai/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Naveen and I am a Chief AI Engineer")])

AIMessage(content='Hello Naveen! It’s great to meet a Chief AI Engineer. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi , My name is Naveen and I am a Chief AI Engineer". Probably they want a response greeting them. We should respond politely, maybe ask how can we help. No policy issues.'}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 85, 'total_tokens': 159, 'completion_time': 0.164122029, 'completion_tokens_details': {'reasoning_tokens': 44}, 'prompt_time': 0.030991916, 'prompt_tokens_details': None, 'queue_time': 0.049845058, 'total_time': 0.195113945}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cb8aa-5b05-78b3-84a5-498ce568679e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_tokens': 74, 'total_tokens': 159, 'output_token_detai

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Naveen and I am a Chief AI Engineer"),
        AIMessage(content="Hello Krish! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='You told me that your name is **Naveen** and that you work as a **Chief AI Engineer**. Sorry for the mix‑up earlier! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user asks: "Hey What\'s my name and what do I do?" They previously said: "Hi , My name is Naveen and I am a Chief AI Engineer". The assistant responded incorrectly (addressed as Krish). Now the user asks to recall name and role. The assistant should correct itself, acknowledge mistake, and provide correct name and role. No policy violation. Should be friendly.'}, response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 152, 'total_tokens': 281, 'completion_time': 0.270630174, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.005741902, 'prompt_tokens_details': None, 'queue_time': 0.156459638, 'total_time': 0.276372076}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c15aa9c1b7', 'service_tier': 'on_demand', 'finish_reaso

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [6]:
config={"configurable":{"session_id":"chat1"}}

In [7]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Naveen and I am a Chief AI Engineer")],
    config=config
)

In [8]:
response.content

'Hello Naveen! It’s great to meet you. How can I assist you today?'

In [9]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='You introduced yourself as **Naveen**.', additional_kwargs={'reasoning_content': 'The user asks "What\'s my name?" They introduced themselves as Naveen. So answer: Naveen.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 117, 'total_tokens': 158, 'completion_time': 0.087368864, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.036845575, 'prompt_tokens_details': None, 'queue_time': 0.068212723, 'total_time': 0.124214439}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cb8aa-ec0c-7c31-af91-15a475a95668-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 41, 'total_tokens': 158, 'output_token_details': {'reasoning': 22}})

In [10]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'I’m sorry—I don’t have any information about your name. If you’d like, you can tell me what you’d like to be called, and I’ll use that going forward.'

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is Arthur")],
    config=config1
)
response.content

'Nice to meet you, Arthur! How can I help you today?'

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is Arthur.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [13]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [15]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Naveen")]})

AIMessage(content='Hello Naveen! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi My name is Naveen". Probably they want a greeting. We should respond politely, maybe ask how can we help. There\'s no disallowed content. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 98, 'total_tokens': 163, 'completion_time': 0.135551195, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.003658339, 'prompt_tokens_details': None, 'queue_time': 0.044953941, 'total_time': 0.139209534}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cb8ab-3064-7531-a5b2-ad53fdf92cfb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 65, 'total_tokens': 163, 'output_token_details': {'reasoning': 40}})

In [16]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [17]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Naveen")],
    config=config
)

response

AIMessage(content='Hello Naveen! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'We need to respond. The user says "Hi My name is Naveen". We should greet and ask how can help. Follow policy: no disallowed content. Just friendly.'}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 98, 'total_tokens': 160, 'completion_time': 0.130428973, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.003806211, 'prompt_tokens_details': None, 'queue_time': 0.153475995, 'total_time': 0.134235184}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cb8ab-4c21-74e2-9cdc-be6d7272602c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 62, 'total_tokens': 160, 'output_token_details': {'reasoning': 37}})

In [18]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Naveen.'

In [19]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [21]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Naveen")],"language":"Tamil"})
response.content

'வணக்கம், நவீன்! உங்களை சந்தித்ததில் மகிழ்ச்சி. இன்று நான் உங்களுக்கு எப்படி உதவலாம்?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [22]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [26]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Naveen")],"language":"Tamil"},
    config=config
)
repsonse.content

'வணக்கம், Naveen! எப்படி இருக்கிறீர்கள்? உங்கள் உதவிக்கு ஏதேனும் தேவையெனில் சொல்லுங்கள்.'

In [27]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Tamil"},
    config=config,
)

In [28]:
response.content

'உங்கள் பெயர்\u202fNaveen\u202fஆகும்.'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [29]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

/home/common/gen-ai/venv/lib/python3.10/site-packages/langchain_core/language_models/base.py:336: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [30]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'I’m not sure which flavor you prefer—could you let me know what your favorite ice‑cream flavor is?'

In [ ]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked "whats 2 + 2" 😊  \n\n\n\n'

In [ ]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [ ]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"As a large language model, I don't have access to past conversations or any personal information about you, including your name.  \n\nIf you'd like to tell me your name, I'd be happy to know! 😊  \n\n"

In [ ]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"As a large language model, I have no memory of past conversations. If you'd like to ask me a math problem, I'm happy to help! 😊  Just let me know what it is. \n\n"